# 🎾 Game Analytics: Unlocking Tennis Data with Sportradar API
### Comprehensive Relational SQL & Data Analytics Walkthrough
**Team Allocation:**
- **Ashish:** Data Extraction (API layer - Sportradar Competitions, Complexes, Doubles Rankings)
- **Nithish:** SQL Database (Relational Schema & 20 Analytical SQL Queries)
- **Dibya:** Streamlit Interactive BI Dashboard & Live Query Engine



In [ ]:
import os
import pymysql
import sqlite3
import pandas as pd
import plotly.express as px
from db_manager import run_query, SQL_QUERIES, get_table_counts

print('Libraries and Database Manager loaded successfully!')



## 1. Database Health Check & Record Counts
Connecting to both MySQL (`tennis_analytics`) and the local SQLite replica (`tennis_analytics.db`).


In [ ]:
counts_mysql = get_table_counts('mysql')
counts_sqlite = get_table_counts('sqlite')

df_counts = pd.DataFrame({
    'Table Name': list(counts_mysql.keys()),
    'MySQL Row Count': list(counts_mysql.values()),
    'SQLite Row Count': list(counts_sqlite.values())
})
df_counts



## 2. Competitions & Category Analysis
### Query 2: Competition Count per Category


In [ ]:
q2_sql = '''SELECT cat.category_name, COUNT(c.competition_id) AS competition_count 
FROM competitions c
JOIN categories cat ON c.category_id = cat.category_id
GROUP BY cat.category_name
ORDER BY competition_count DESC;'''

df_q2, ms, _ = run_query(q2_sql, 'mysql')
print(f'Execution Time: {ms:.2f} ms')
display(df_q2.head(10))

fig_q2 = px.bar(df_q2, x='category_name', y='competition_count', color='competition_count',
                title='Competition Distribution by Tennis Category', template='plotly_dark')
fig_q2.show()



### Query 6: Competition Type Distribution by Category


In [ ]:
q6_sql = '''SELECT cat.category_name, c.type, COUNT(c.competition_id) AS type_count 
FROM competitions c
JOIN categories cat ON c.category_id = cat.category_id
GROUP BY cat.category_name, c.type
ORDER BY cat.category_name, type_count DESC;'''

df_q6, _, _ = run_query(q6_sql, 'mysql')
display(df_q6.head(10))



### Query 5: Parent & Sub-Competition Hierarchies (Business Use Case 1)


In [ ]:
q5_sql = '''SELECT parent.competition_id AS parent_competition, child.competition_name AS sub_competition 
FROM competitions child
LEFT JOIN competitions parent ON child.parent_id = parent.competition_id
WHERE child.parent_id IS NOT NULL;'''

df_q5, _, _ = run_query(q5_sql, 'mysql')
df_q5



## 3. Venues & Complexes Infrastructure
### Query 10: Chilean Venue Details


In [ ]:
q10_sql = '''SELECT venue_id, venue_name, city_name, country_name, timezone
FROM Venues
WHERE country_name = 'Chile';'''

df_q10, _, _ = run_query(q10_sql, 'mysql')
display(df_q10.head(10))



### Query 12: Complexes with More than One Venue


In [ ]:
q12_sql = '''SELECT c.complex_name, COUNT(v.venue_id) AS venue_count
FROM Complexes c
JOIN Venues v ON c.complex_id = v.complex_id
GROUP BY c.complex_name
HAVING COUNT(v.venue_id) > 1
ORDER BY venue_count DESC
LIMIT 10;'''

df_q12, _, _ = run_query(q12_sql, 'mysql')
df_q12



## 4. Competitor Rankings & Country Strength
### Query 16: Top 5 World Doubles Competitors


In [ ]:
q16_sql = '''SELECT c.name AS competitor_name, c.country, cr.rank, cr.points
FROM Competitors c
JOIN Competitor_Rankings cr ON c.competitor_id = cr.competitor_id
WHERE cr.rank <= 5
ORDER BY cr.rank ASC;'''

df_q16, _, _ = run_query(q16_sql, 'mysql')
df_q16



### Query 17: Stable Rank Competitors (Movement = 0)


In [ ]:
q17_sql = '''SELECT c.name AS competitor_name, c.country, cr.rank, cr.points
FROM Competitors c
JOIN Competitor_Rankings cr ON c.competitor_id = cr.competitor_id
WHERE cr.movement = 0
ORDER BY cr.rank ASC;'''

df_q17, _, _ = run_query(q17_sql, 'mysql')
display(df_q17.head(10))



### Query 18: Total Ranking Points for Croatia


In [ ]:
q18_sql = '''SELECT c.country, SUM(cr.points) AS total_points
FROM Competitors c
JOIN Competitor_Rankings cr ON c.competitor_id = cr.competitor_id
WHERE c.country = 'Croatia'
GROUP BY c.country;'''

df_q18, _, _ = run_query(q18_sql, 'mysql')
df_q18



### Query 20: Highest Points Competitor in Current Week


In [ ]:
q20_sql = '''SELECT c.name AS competitor_name, c.country, cr.rank, cr.points
FROM Competitors c
JOIN Competitor_Rankings cr ON c.competitor_id = cr.competitor_id
WHERE cr.points = (
    SELECT MAX(points)
    FROM Competitor_Rankings
);'''

df_q20, _, _ = run_query(q20_sql, 'mysql')
df_q20

